In [99]:
%load_ext autoreload

In [100]:
%autoreload 2

In [33]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from dataset import TranslationDataset, PAD_ID, collate_fn
from datasets import load_from_disk

In [118]:
import model as M
from model import Transformer, DecoderBlock, Head, EncoderBlock, vocab_size, Encoder

In [119]:
ds = load_from_disk("data/tokenized")
ds.shape

{'train': (507684, 4), 'validation': (10370, 4), 'test': (36380, 4)}

In [120]:
train_ds = TranslationDataset(ds["train"])
val_ds = TranslationDataset(ds["validation"])
test_ds = TranslationDataset(ds["test"])

In [122]:
enc = EncoderBlock(n_embd=256, n_head=4)
enc.eval()
x = torch.randn(2, 5, 256)
pad_mask = torch.tensor([[False]*5, [False, False, False, True, True]])
out_block = enc(x, pad_mask)
print(out_block.shape)

torch.Size([2, 5, 256])


In [123]:
x2 = x.clone()
x2[1, 3:] = torch.randn(2, 256)
out_block2 = enc(x2, pad_mask)
print(torch.allclose(out_block[1, :3], out_block2[1, :3], atol=1e-6))

True


In [124]:
loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
src, tgt_in, tgt_out = next(iter(loader))
enc = Encoder()
enc.eval()
with torch.no_grad():
    x, pad_mask = enc(src)
print(x.shape, pad_mask.shape)
print((pad_mask == (src==PAD_ID)).all())

torch.Size([64, 13, 256]) torch.Size([64, 13])
tensor(True)


In [125]:
src_ext = F.pad(src, (0, 5), value=PAD_ID)
with torch.no_grad():
    x_ext, _ = enc(src_ext)
S = src.shape[1]
print(torch.allclose(x[~pad_mask], x_ext[:, :S][~pad_mask], atol=1e-5))

True


In [126]:
head = Head(head_size=64, n_embd=256)
head.eval()
x = torch.randn(2, 5, 256)
out = head(x, x, None, causal=True)
x2 = x.clone()
x2[:, 3:] = torch.randn(2, 256)
out2 = head(x2, x2, None, causal=True)
print(torch.allclose(out[:, :3], out2[:, :3], atol=1e-6))

True


In [127]:
xq = torch.randn(2, 3, 256)
xkv = torch.randn(2, 7, 256)
kpm = torch.tensor([[False]*7, [False]*4 + [True]*3])
o = head(xq, xkv, kpm)
print(o.shape)                     # (2, 3, 64)

xkv2 = xkv.clone()
xkv2[1, 4:] = torch.randn(3, 256)  # меняем только замаскированные позиции
print(torch.allclose(o, head(x_kv=xkv2, x_q=xq, key_pad_mask=kpm), atol=1e-6))   # True

torch.Size([2, 3, 64])
True


In [128]:
blk = DecoderBlock(256, 4)
blk.eval()
x = torch.randn(2, 4, 256)
enc_out = torch.randn(2, 6, 256)
tpm = torch.zeros(2, 4, dtype=torch.bool)
spm = torch.tensor([[False]*6, [False]*4 + [True]*2])

print(x.shape, enc_out.shape, tpm.shape, spm.shape)
o = blk(x, enc_out, tpm, spm)
print(o.shape)

# 1) замаскированные позиции источника не влияют
e2 = enc_out.clone(); e2[1, 4:] = torch.randn(2, 256)
print(torch.allclose(o, blk(x, e2, tpm, spm), atol=1e-6))        

# 2) causal: меняем будущие позиции x, начало не должно измениться
x2 = x.clone(); x2[:, 2:] = torch.randn(2, 2, 256)
print(torch.allclose(o[:, :2], blk(x2, enc_out, tpm, spm)[:, :2], atol=1e-6))

torch.Size([2, 4, 256]) torch.Size([2, 6, 256]) torch.Size([2, 4]) torch.Size([2, 6])
torch.Size([2, 4, 256])
True
True


In [129]:
model = Transformer()
model.eval()
with torch.no_grad():
    logits = model(src, tgt_in)
print(logits.shape)                                   # (64, T, 16000)
loss = F.cross_entropy(logits.view(-1, vocab_size), tgt_out.view(-1), ignore_index=PAD_ID)
print(loss.item())                                    # ожидаем около 9.7
print(sum(p.numel() for p in model.parameters()))     # число параметров

torch.Size([64, 18, 16000])
9.870845794677734
17893248


In [130]:
i = 3
tgt2 = tgt_in.clone()
tgt2[:, i] = torch.where(tgt_in[:, i] != PAD_ID, torch.tensor(100), tgt_in[:, i])
with torch.no_grad():
    l1 = model(src, tgt_in)
    l2 = model(src, tgt2)
print(torch.allclose(l1[:, :i], l2[:, :i], atol=1e-5))     # True
print(torch.allclose(l1[:, i:], l2[:, i:], atol=1e-5))     # False

True
False


In [131]:
M.dropout = 0.0
model = M.Transformer()
small = torch.utils.data.Subset(train_ds, range(0, len(train_ds), 25000))
src, tgt_in, tgt_out = next(iter(DataLoader(small, batch_size=20, collate_fn=collate_fn)))

opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train()
for step in range(300):
    logits = model(src, tgt_in)
    loss = F.cross_entropy(logits.view(-1, vocab_size), tgt_out.view(-1), ignore_index=PAD_ID)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 25 == 0:
        print(step, loss.item())

0 9.816341400146484
25 0.073216512799263
50 0.008620940148830414
75 0.005260726436972618
100 0.004211451392620802
125 0.003563739825040102
150 0.0030785398557782173
175 0.0026946323923766613
200 0.0023816393222659826
225 0.0021221584174782038
250 0.0019039169419556856
275 0.0017181001603603363
